## Gold Layer - Business-Ready Tables

 build business-ready tables from the cleaned silver tables. Each gold table answers one business question and is designed to feed the dashboard directly.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_finance.gold_layer;

In [0]:

%sql
CREATE OR REPLACE TABLE medical_finance.gold_layer.spend_by_vendor AS
-- How much do we spend with each vendor?
SELECT
  seller                       AS vendor,
  currency,
  COUNT(*)                     AS invoice_count,
  ROUND(SUM(total_amount), 2)  AS total_spend,
  ROUND(AVG(total_amount), 2)  AS avg_invoice_amount,
  MIN(invoice_date)            AS first_invoice_date,
  MAX(invoice_date)            AS last_invoice_date
FROM medical_finance.silver_layer.invoices_cleaned
GROUP BY seller, currency
ORDER BY total_spend DESC;

In [0]:
%sql
-- How does our spending change each month?
CREATE OR REPLACE TABLE medical_finance.gold_layer.monthly_spend AS
WITH monthly AS (
  SELECT
    DATE_TRUNC('MONTH', invoice_date) AS month,
    currency,
    ROUND(SUM(total_amount), 2)       AS total_spend,
    COUNT(*)                          AS invoice_count
  FROM medical_finance.silver_layer.invoices_cleaned
  WHERE invoice_date IS NOT NULL
  GROUP BY DATE_TRUNC('MONTH', invoice_date), currency
)
SELECT
  month,
  currency,
  total_spend,
  invoice_count,
  LAG(total_spend) OVER (PARTITION BY currency ORDER BY month) AS previous_month_spend,
  ROUND(100 * (total_spend - LAG(total_spend) OVER (PARTITION BY currency ORDER BY month))
        / LAG(total_spend) OVER (PARTITION BY currency ORDER BY month), 2) AS growth_pct
FROM monthly;

In [0]:
%sql
-- Does each invoice match its purchase order?
CREATE OR REPLACE TABLE medical_finance.gold_layer.po_invoice_reconciliation AS
SELECT
  po.po_number,
  po.vendor,
  po.po_date,
  po.total_amount                              AS po_total,
  inv.invoice_number,
  inv.invoice_date,
  inv.total_amount                             AS invoice_total,
  ROUND(inv.total_amount - po.total_amount, 2) AS difference,
  CASE
    WHEN inv.invoice_number IS NULL                         THEN 'No invoice yet'
    WHEN ABS(inv.total_amount - po.total_amount) <= 0.01    THEN 'Matched'
    WHEN inv.total_amount > po.total_amount                 THEN 'Overbilled'
    ELSE 'Underbilled'
  END AS reconciliation_status
FROM medical_finance.silver_layer.purchase_orders_cleaned AS po
LEFT JOIN medical_finance.silver_layer.invoices_cleaned AS inv
  ON po.po_number = inv.po_number;